# dfthcsos: train, verify and package (commit mode)

Designed for **Save Version → Save & Run All (Commit)**: it runs on Kaggle's servers even with the browser closed,
and everything it writes to `/kaggle/working` stays in the version's **Output** tab.

Inputs (right panel, **+ Add Input**): the competition, the *Verifiable Circuit Compilation Starter Kit* dataset and
your `dfthcsos-toolkit` dataset. Settings: **GPU** accelerator, **Internet** on.

After each experiment the best result so far is packaged as `/kaggle/working/submission.zip`, so a failure
in a later experiment never loses an earlier one.

In [ ]:
# Settings. EPOCH_SCALE < 1 shortens every run (only for quick tests); keep it at 1.0 for real runs.
EPOCH_SCALE = 1.0

In [ ]:
# 1. Locate the inputs and prepare the work folders
import functools, json, os, shutil, subprocess, sys, traceback, zipfile
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

def find(pattern, what):
    hits = list(INPUT.glob(pattern))
    if not hits:
        raise FileNotFoundError(f'{what} not found: add it with "+ Add Input" (right panel).')
    return hits[0]

OFFICIAL = INPUT / 'competitions' / 'google-verifiable-quantum-circuit-compilation'
COMPETITION = OFFICIAL if (OFFICIAL / 'verifiers').is_dir() else \
    find('**/verifiers/dfthcsos/verify.py', 'the competition data').parents[2]
KIT_SRC = find('**/starter_kit/starter', 'the starter kit').parent
TOOLKIT_SRC = find('**/run_experiments.py', 'the dfthcsos-toolkit dataset').parent
print('competition :', COMPETITION)
print('starter kit :', KIT_SRC)
print('toolkit     :', TOOLKIT_SRC)

# The toolkit is copied to a writable folder.
TOOLKIT = WORK / 'toolkit'
shutil.copytree(TOOLKIT_SRC, TOOLKIT, dirs_exist_ok=True)

# The trainer looks for the FCIDUMP and the fixed shifts in the folder above starter_kit/.
KIT_PARENT = WORK / 'kit'; KIT_PARENT.mkdir(exist_ok=True)
for link, target in [('starter_kit', KIT_SRC),
                     ('FeMoCo76o113e.fcidump', COMPETITION / 'verifiers/dfthcsos/FeMoCo76o113e.fcidump'),
                     ('FeMoCo76o113e.fixed_shifts.npz', COMPETITION / 'verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz')]:
    if not (KIT_PARENT / link).exists():
        (KIT_PARENT / link).symlink_to(target)

os.environ.update({'COMPETITION_DIR': str(COMPETITION),
                   'KIT_DIR': str(KIT_PARENT / 'starter_kit'),
                   'WORK_DIR': str(WORK)})
print('ok')

In [ ]:
# 2. Check the GPU and the libraries
import importlib.util
if importlib.util.find_spec('optax') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optax'], check=True)
import jax
print('JAX backend:', jax.default_backend())  # should be 'gpu'

In [ ]:
# 3. Load the toolkit and define the helpers
sys.path.insert(0, str(TOOLKIT))
os.chdir(TOOLKIT)
import run_experiments as RX
import starter.task1_factorization.train_df_thc_qaggle as TQ
from dfthc import train_df_thc_sga as TS

REF = RX.lab.ref()
SORTED_SEED = TOOLKIT / 'FeMoCo76o113e_65_60_60_sorted.npz'


def train(name, seed=None, lr_final=1e-6, **cfg):
    """Train, build and verify one config; the learning rate decays exponentially from lr to lr_final.

    Errors are printed instead of raised, so later cells (and packaging) still run.
    """
    cfg['epochs'] = max(20, int(cfg['epochs'] * EPOCH_SCALE))
    RX.SEED = Path(seed) if seed and Path(seed).exists() else SORTED_SEED
    print(f'=== {name}: seed {RX.SEED.name}, {cfg}')
    TQ.Train = functools.partial(TS.Train, learning_rate_final=lr_final)
    try:
        return RX.run(name, cfg, REF)
    except Exception:
        traceback.print_exc()
        return None


def package():
    """Write submission.zip from the best verified result so far (reference circuits for invsqrt and dqi)."""
    results = WORK / 'results.jsonl'
    rows = [json.loads(line) for line in open(results)] if results.exists() else []
    for r in rows:
        print(f"{r['name']:14s} status={str(r.get('status')):20s} brot={r.get('brot')} "
              f"improvement={r.get('improvement', float('nan')):.4f}")
    passed = [r for r in rows if r.get('status') == 'PASS']
    if not passed:
        print('No verified result yet: submission.zip not written.')
        return
    best = max(passed, key=lambda r: r['improvement'])
    sample = COMPETITION / 'sample_submission'
    with zipfile.ZipFile(WORK / 'submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for name in ('invsqrt.kmb', 'dqi.kmb'):
            zf.write(sample / name, name)
        for path in sorted(Path(best['folder']).iterdir()):
            zf.write(path, f'dfthcsos/{path.name}')
    print(f"submission.zip <- {best['name']} (improvement x{best['improvement']:.4f})")

## Experiment 1: R = 60 (about 70 min; reached x1.081 before)

In [ ]:
train('R60_decay', rank=60, bases=60, copies=60, epochs=50_000,
      lr=1e-4, lr_final=1e-6, rho=1e-4, epsilon_dfthc=0.015)
package()

## Experiment 2: R = 55, seeded from the R = 60 result (about 1 h 45 min; x1.17 if it passes)

In [ ]:
train('R55_decay', seed=WORK / 'runs' / 'R60_decay.npz', rank=55, bases=60, copies=60, epochs=80_000,
      lr=1e-4, lr_final=1e-6, rho=1e-4, epsilon_dfthc=0.015)
package()

## Clean up
Only `submission.zip`, `results.jsonl`, the trained factorizations (`runs/`) and the winning build are worth keeping.

In [ ]:
os.chdir(WORK)
# Remove the copied toolkit, the links and the cache so the Output tab stays small
for path in (WORK / 'toolkit', WORK / 'kit', WORK / 'femoco76_ref_cache.npz'):
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
print(sorted(p.name for p in WORK.iterdir()))